In [ ]:
# ============================================================================
# DATASET v2 — GERAÇÃO COM THRESHOLD ECOLÓGICO DE 5 ANOS
# ============================================================================
# Motivação:
#   O threshold anterior (10 anos) era uma decisão de balanceamento.
#   O threshold de 5 anos reflete o processo real de limpeza do Cerrado:
#   fazendeiros capitalizados fazem N→P→S em 12-18 meses;
#   o prazo normal é 2-4 anos; máximo do processo intencional: 5 anos.
#
# Definições:
#   T = primeiro ano em que o pixel aparece como classe 15 (pastagem)
#       após vegetação nativa (classe 3=floresta ou 4=savana)
#
#   Label=1 (rápido): pixel atinge classe 39 (soja) dentro de ≤5 anos
#       após T. Pode passar por classes 21 (mosaico) ou 41 (outros
#       cultivos) no caminho — essas são classes de transição aceitas.
#
#   Label=0 (lento/nunca): pixel não atinge 39 dentro de 5 anos após T.
#       Inclui pixels que passaram por 21/41 mas voltaram para pastagem.
#
#   Censurado (excluído): T > 2019 — menos de 5 anos de observação
#       disponível (dados vão até 2024).
#
# Fonte: rasters MapBiomas Collection 10
#   brazil_coverage_{ano}_Cerrado.tif (1985-2024)
# ============================================================================

import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from pathlib import Path
from tqdm.notebook import tqdm
from datetime import datetime
import json

# ── Configuração ─────────────────────────────────────────────────────────────
DATA_DIR  = r"D:\Projetos\Cerrado\LULC"
PATTERN   = "brazil_coverage_{ano}_Cerrado.tif"
BASE_DIR  = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
OUT_DIR   = BASE_DIR / "dataset_v2"
OUT_DIR.mkdir(exist_ok=True, parents=True)

YEARS     = list(range(1985, 2025))   # 1985–2024

# Classes
NATIVE    = {3, 4, 11, 12}   # floresta, savana, wetland, campo nativo
PASTURE   = 15
SOY       = 39
TRANSITION = {21, 41}        # mosaico agrícola, outros cultivos temporários

# Threshold ecológico
THRESHOLD_ANOS = 5           # ≤5 anos após T → label=1
T_MAX = 2019                 # T > 2019 → censurado (menos de 5 anos obs.)

# Dataset de referência para reusar pixels
# (usamos o split espacial existente como base de coordenadas)
SPLIT_DIR = BASE_DIR / "spatial_split"

print(f"✅ Configuração Dataset v2")
print(f"   Threshold: {THRESHOLD_ANOS} anos")
print(f"   T máximo:  {T_MAX} (T > {T_MAX} → censurado)")
print(f"   Label=1:   39 dentro de ≤{THRESHOLD_ANOS} anos, via 21/41 aceitos")
print(f"   Label=0:   39 não atingido em {THRESHOLD_ANOS} anos")
print(f"   Output:    {OUT_DIR}")

In [ ]:
# ============================================================================
# FUNÇÕES CORE
# ============================================================================

def ler_pixel(year, row, col):
    path = os.path.join(DATA_DIR, PATTERN.format(ano=year))
    with rasterio.open(path) as ds:
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)


def encontrar_T(row, col):
    """
    Encontra T = primeiro ano em que o pixel aparece como pastagem (15)
    após vegetação nativa (3, 4, 11, 12).

    Lógica:
      - Percorre a série temporal cronologicamente
      - Registra se o pixel estava em nativa no ano anterior
      - T = primeiro ano com classe 15 precedido de nativa

    Retorna: T (int) ou None se não encontrou transição N→P
    """
    prev_native = False
    for year in YEARS:
        v = ler_pixel(year, row, col)
        if v == 255: v = 0   # nodata
        if v in NATIVE:
            prev_native = True
        elif v == PASTURE and prev_native:
            return year
        elif v not in NATIVE and v != PASTURE:
            # Classe agrícola diretamente — reset
            prev_native = False
    return None


def calcular_label(row, col, T):
    """
    Calcula label dado T.

    Lógica:
      - Observa os THRESHOLD_ANOS anos após T
      - Aceita 21 e 41 como classes de transição
      - Label=1 se 39 é atingido dentro do prazo
      - Label=0 se não atingiu 39 (mesmo passando por 21/41)

    Retorna:
      (label, ano_conversao) onde label ∈ {0, 1}
      ano_conversao = ano em que 39 foi atingido (ou None se label=0)
    """
    anos_observacao = [y for y in YEARS if T < y <= T + THRESHOLD_ANOS]

    for year in anos_observacao:
        v = ler_pixel(year, row, col)
        if v == 255: v = 0

        if v == SOY:
            return 1, year   # converteu para soja dentro do prazo

        elif v in TRANSITION:
            # Classe de transição aceita — continuar observando
            continue

        elif v == PASTURE:
            # Voltou para pastagem — pode ainda converter
            continue

        else:
            # Outra classe (nativa, urbano, etc.) — improvável converter
            # mas ainda observar o prazo completo
            continue

    return 0, None   # não converteu dentro do prazo


print("✅ Funções definidas")
print()
print("Lógica de label:")
print(f"  Janela: T+1 a T+{THRESHOLD_ANOS}")
print(f"  Aceitos como transição: classes {TRANSITION}")
print(f"  Alvo: classe {SOY} (soja)")

In [ ]:
# ============================================================================
# RECALCULAR LABELS PARA PIXELS EXISTENTES
# ============================================================================
# Reutiliza os pixels do dataset v1 (split espacial existente)
# mas recalcula T e label do zero a partir dos rasters.
# Isso garante consistência total com a nova definição.

# Carregar todos os pixels do split espacial
dfs = []
for split in ['train', 'val', 'test']:
    df_s = pd.read_csv(SPLIT_DIR / f"spatial_split_{split}.csv")
    df_s['split'] = split
    dfs.append(df_s)

df_all = pd.concat(dfs, ignore_index=True)
print(f"Pixels disponíveis: {len(df_all):,}")
print(f"Colunas: {df_all.columns.tolist()}")

# Recalcular para cada pixel
print(f"\nRecalculando T e label para {len(df_all):,} pixels...")
print("⚠️  Lê rasters pixel a pixel — pode demorar 2-4 horas")
print("   Progresso salvo a cada 500 pixels")
print()

results = []
n_label1 = n_label0 = n_censurado = n_no_T = 0

CHECKPOINT_PATH = OUT_DIR / "recalc_checkpoint.csv"

# Retomar de checkpoint se existir
start_idx = 0
if CHECKPOINT_PATH.exists():
    df_check = pd.read_csv(CHECKPOINT_PATH)
    results   = df_check.to_dict('records')
    start_idx = len(results)
    print(f"▶ Retomando do checkpoint: {start_idx:,} pixels já processados")

for idx in tqdm(range(start_idx, len(df_all)), desc="Recalculando labels"):
    r   = df_all.iloc[idx]
    row = int(r['row'])
    col = int(r['col'])
    split = r['split']
    hex_id    = r.get('hex_id', None)
    class8590 = r.get('Class8590', None)

    # Recalcular T do zero
    T_novo = encontrar_T(row, col)

    if T_novo is None:
        n_no_T += 1
        results.append({
            'row': row, 'col': col,
            'T': None, 'label': None,
            'ano_conversao': None,
            'status': 'no_T',
            'split': split,
            'hex_id': hex_id,
            'Class8590': class8590
        })
        continue

    # Censurado: T > T_MAX
    if T_novo > T_MAX:
        n_censurado += 1
        results.append({
            'row': row, 'col': col,
            'T': T_novo, 'label': None,
            'ano_conversao': None,
            'status': 'censored',
            'split': split,
            'hex_id': hex_id,
            'Class8590': class8590
        })
        continue

    # Calcular label
    label, ano_conv = calcular_label(row, col, T_novo)

    if label == 1:
        n_label1 += 1
        status = 'rapid'
    else:
        n_label0 += 1
        status = 'slow'

    results.append({
        'row': row, 'col': col,
        'T': T_novo, 'label': label,
        'ano_conversao': ano_conv,
        'status': status,
        'split': split,
        'hex_id': hex_id,
        'Class8590': class8590
    })

    # Checkpoint a cada 500
    if (idx + 1) % 500 == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)

# Salvar checkpoint final
pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)

print(f"\n✅ Recálculo concluído!")
print(f"   label=1 (rápido):   {n_label1:,}")
print(f"   label=0 (lento):    {n_label0:,}")
print(f"   Censurados (T>2019): {n_censurado:,}")
print(f"   Sem T (sem N→P):    {n_no_T:,}")

In [ ]:
# ============================================================================
# CONSTRUIR DATASET v2 FINAL
# ============================================================================

df_results = pd.read_csv(CHECKPOINT_PATH)

print("=" * 60)
print("DISTRIBUIÇÃO COMPLETA")
print("=" * 60)
print(df_results['status'].value_counts())

# Dataset válido: apenas pixels com label definido
df_valid = df_results[df_results['label'].notna()].copy()
df_valid['label'] = df_valid['label'].astype(int)
df_valid['T']     = df_valid['T'].astype(int)

print(f"\nDataset válido: {len(df_valid):,} pixels")
print(f"  label=1 (≤5 anos): {(df_valid['label']==1).sum():,} "
      f"({100*(df_valid['label']==1).mean():.1f}%)")
print(f"  label=0 (>5 anos): {(df_valid['label']==0).sum():,} "
      f"({100*(df_valid['label']==0).mean():.1f}%)")

print(f"\nRange de T: {df_valid['T'].min()} – {df_valid['T'].max()}")
print(f"T mediana: {df_valid['T'].median():.0f}")

# Distribuição de T
print(f"\nDistribuição de T por label:")
print(df_valid.groupby('label')['T'].describe()[['min','25%','50%','75%','max']]
      .round(0).to_string())

# Verificar balanço — se muito desequilibrado, alertar
ratio = (df_valid['label']==1).mean()
if ratio < 0.2 or ratio > 0.8:
    print(f"\n⚠️  Dataset desequilibrado ({ratio:.1%} positivos)")
    print(f"   Considerar oversampling da classe minoritária")
else:
    print(f"\n✅ Balanceamento razoável ({ratio:.1%} positivos)")

# Distribuição por split
print(f"\nDistribuição por split:")
print(df_valid.groupby('split').agg(
    n=('label','count'),
    n_label1=('label','sum'),
    pct_label1=('label','mean')
).round(3).to_string())

# Ano de conversão — distribuição
df_conv = df_valid[df_valid['ano_conversao'].notna()]
if len(df_conv) > 0:
    print(f"\nAno de conversão (label=1):")
    print(df_conv['ano_conversao'].value_counts().sort_index().to_string())

In [ ]:
# ============================================================================
# SALVAR DATASET v2 E SPLITS
# ============================================================================

import matplotlib.pyplot as plt

# Dataset completo
out_full = OUT_DIR / "dataset_v2_full.csv"
df_valid.to_csv(out_full, index=False)
print(f"✅ Dataset completo: {out_full.name} ({len(df_valid):,} pixels)")

# Splits separados (mantendo o split espacial hexagonal do v1)
for split in ['train', 'val', 'test']:
    df_split = df_valid[df_valid['split'] == split].reset_index(drop=True)
    # Colunas para o modelo (compatível com LULCDatasetV3)
    cols = ['row', 'col', 'T', 'label', 'hex_id', 'Class8590']
    cols_out = [c for c in cols if c in df_split.columns]
    out_path = OUT_DIR / f"dataset_v2_{split}.csv"
    df_split[cols_out].to_csv(out_path, index=False)
    n1 = (df_split['label']==1).sum()
    n0 = (df_split['label']==0).sum()
    print(f"✅ {split}: {len(df_split):,} pixels "
          f"(label=1: {n1:,} = {100*n1/len(df_split):.1f}%)")

# Metadata
meta = {
    'version': 'dataset_v2',
    'created': datetime.now().strftime("%Y-%m-%d %H:%M"),
    'threshold_anos': THRESHOLD_ANOS,
    'T_max': T_MAX,
    'label_1_definition': f'Pixel atinge classe 39 em ≤{THRESHOLD_ANOS} anos após T, '
                           f'podendo passar por classes {list(TRANSITION)}',
    'label_0_definition': f'Pixel não atinge classe 39 em {THRESHOLD_ANOS} anos após T',
    'censored_definition': f'T > {T_MAX} (menos de {THRESHOLD_ANOS} anos de observação)',
    'T_definition': 'Primeiro ano com classe 15 após classes nativas (3,4,11,12)',
    'source': 'MapBiomas Collection 10, rasters recalculados do zero',
    'split_method': 'Hexagon-stratified (mantido do dataset v1)',
    'ecological_basis': (
        'Threshold de 5 anos baseado no processo de limpeza do Cerrado: '
        'fazendeiros capitalizados: 12-18 meses; prazo normal: 2-4 anos; '
        'máximo do processo intencional de conversão: 5 anos'
    ),
    'total_valid': int(len(df_valid)),
    'n_label1': int((df_valid['label']==1).sum()),
    'n_label0': int((df_valid['label']==0).sum()),
    'splits': {
        s: int((df_valid['split']==s).sum()) for s in ['train','val','test']
    }
}

with open(OUT_DIR / 'dataset_v2_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"✅ Metadata salvo")

# Gráfico de distribuição
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribuição de T por label
for lbl, color, name in [(1,'#c0392b','Rápido (≤5 anos)'),
                          (0,'#2980b9','Lento (>5 anos)')]:
    sub = df_valid[df_valid['label']==lbl]['T']
    axes[0].hist(sub, bins=20, alpha=0.6, color=color, label=f'{name} (n={len(sub)})')
axes[0].set_xlabel('T (ano entrada em pastagem)')
axes[0].set_ylabel('N pixels')
axes[0].set_title('Distribuição de T por label', fontweight='bold')
axes[0].legend()

# Distribuição de ano de conversão
df_conv = df_valid[df_valid['ano_conversao'].notna()]
if len(df_conv) > 0:
    axes[1].hist(df_conv['ano_conversao'], bins=20,
                 color='#c0392b', alpha=0.8)
    axes[1].set_xlabel('Ano de conversão para soja')
    axes[1].set_ylabel('N pixels')
    axes[1].set_title('Distribuição do ano de conversão\n(label=1)', fontweight='bold')

# Balanço por split
split_stats = df_valid.groupby('split').agg(
    pct_label1=('label','mean')).reset_index()
axes[2].bar(split_stats['split'], split_stats['pct_label1']*100,
            color=['#27ae60','#f39c12','#e74c3c'])
axes[2].axhline(y=50, color='black', linestyle='--', alpha=0.5)
axes[2].set_ylabel('% label=1 (rápido)')
axes[2].set_title('Balanço por split', fontweight='bold')
axes[2].set_ylim(0, 100)

plt.suptitle('Dataset v2 — Threshold 5 anos', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'dataset_v2_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'='*60}")
print(f"DATASET v2 COMPLETO")
print(f"{'='*60}")
print(f"📁 {OUT_DIR}")
print(f"   dataset_v2_full.csv       ({len(df_valid):,} pixels)")
print(f"   dataset_v2_train.csv")
print(f"   dataset_v2_val.csv")
print(f"   dataset_v2_test.csv")
print(f"   dataset_v2_metadata.json")
print(f"   dataset_v2_distribution.png")
print(f"\n🎯 Próximo passo: registrar no OSF e treinar modelo com novo threshold")